# Exploratory Data Analysis

In [ ]:
# Import Python libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Read dataset into dataframe
df = pd.read_csv('data/flash_floods_ky_2015_2025.csv')

In [ ]:
# Read first rows of dataset
df.head()

In [ ]:
# Read last rows of dataset
df.tail()

In [ ]:
# Check data type and missing values
df.info()

print(df.columns.tolist())

In [ ]:
# See the dimensions of the dataset
df.shape

In [ ]:
# Find categorical variables using .select_dtypes()
df.select_dtypes(include='object').columns

In [ ]:
# Explore value counts for categorial variables
df.value_counts(subset=['FLOOD_CAUSE']),
df.value_counts(subset=['CZ_NAME_STR'])

print(df['FLOOD_CAUSE'].value_counts())
print(df['CZ_NAME_STR'].value_counts())

In [ ]:
# Explore numberical columns
df.describe()

In [ ]:
# Filter rows using .loc[] ,.iloc[] and/or boolean indexing
df.loc[df['CZ_NAME_STR'] == 'Kentucky']
df.iloc[0:5]

In [ ]:
# Sort values using .sort_values()
df.sort_values(by='BEGIN_DATE', ascending=True)

In [ ]:
# Rename columns using .rename()
df.rename(columns={'CZ_NAME_STR': 'COUNTY_NAME'}, inplace=True)

print(df.columns.tolist())

In [ ]:
# Drop columns using .drop()
df.drop(columns=['MAGNITUDE', 'MAGNITUDE_TYPE', 'TOR_F_SCALE', 'TOR_LENGTH', 'TOR_WIDTH'], inplace=True)

print(df.columns.tolist())

In [ ]:
# Fix data types using .astype()
df = df.astype({'BEGIN_DATE': 'datetime64[ns]'})
df = df.astype({'END_DATE': 'datetime64[ns]'})
df = df.astype({'EVENT_ID': 'string'})
df = df.astype({'BEGIN_TIME': 'string'})
df = df.astype({'END_TIME': 'string'})
df = df.astype({'EPISODE_ID': 'string'})
df = df.astype({'CZ_FIPS': 'string'})
df = df.astype({'EVENT_NARRATIVE': 'string'})
df = df.astype({'EPISODE_NARRATIVE': 'string'})

print(df.dtypes)

In [ ]:
# Identify missing values using .isnull() and/or .notnull()
df.isnull().sum()

In [ ]:
# View rows with missing values
df[df.isnull().any(axis=1)] 

print(df[df['EVENT_NARRATIVE'].isnull()])

In [ ]:
# Replace missing values in EVENT_NARRATIVE with "Unknown"
df["EVENT_NARRATIVE"] = df["EVENT_NARRATIVE"].fillna("Unknown")

print(df["EVENT_NARRATIVE"].isna().sum())

In [ ]:
# Check for duplicate rows using .duplicated()
df.duplicated().sum()

In [ ]:
# Create YEAR column
df["YEAR"] = df["BEGIN_DATE"].dt.year

df = df.astype({'YEAR': 'int64'})

print(df["YEAR"].value_counts())

In [ ]:
df.describe()

In [ ]:
# Use .groupby() to explore number of flash flooding events by county
county_counts = (
    df.groupby("COUNTY_NAME")
      .size()
      .reset_index(name="Total_Events")
      .sort_values("Total_Events", ascending=False)
)

print(county_counts.head(10))

In [ ]:
# Use .groupby() to explore total property damage by county
property_damage = (
    df.groupby("COUNTY_NAME")["DAMAGE_PROPERTY_NUM"]
      .sum()
      .reset_index()
      .sort_values("DAMAGE_PROPERTY_NUM", ascending=False)
)

print(property_damage.head(10))

In [ ]:
# Use .groupby() to explore total deaths and injuries by county
human_impact = (
    df.groupby("COUNTY_NAME")[["DEATHS_DIRECT", "INJURIES_DIRECT"]]
      .sum()
      .reset_index()
      .sort_values("DEATHS_DIRECT", ascending=False)
)

print(human_impact.head(10))

In [ ]:
# Use .groupby() to explore how many flash flooding events were reported from each source
source_counts = (
    df.groupby("SOURCE")
      .size()
      .reset_index(name="Total_Events")
      .sort_values("Total_Events", ascending=False)
)

print(source_counts)

In [ ]:
# Use .groupby() to explore the top 5 dates that had the most flash flooding events
date_counts = (
    df.groupby("BEGIN_DATE")
      .size()
      .reset_index(name="TOTAL_EVENTS")
      .sort_values("TOTAL_EVENTS", ascending=False)
)

print(date_counts.head(5))

In [ ]:
# Use .groupby() to explore the counties and cities that experienced flash flooding events on the top 5 dates. Show the dates and the total number of events for each county on those dates.
top_dates = date_counts.head(5)["BEGIN_DATE"]
county_city_date_counts = (
    df[df["BEGIN_DATE"].isin(top_dates)]
      .groupby(["BEGIN_DATE", "COUNTY_NAME", "BEGIN_LOCATION"])
      .size()
      .reset_index(name="TOTAL_EVENTS")
      .sort_values(["BEGIN_DATE", "TOTAL_EVENTS"], ascending=[True, False])
)

print(county_city_date_counts)

In [ ]:
# Immport data into csv file
county_city_date_counts.to_csv("data/output/county_city_date_counts.csv", index=False)

In [ ]:
# Sort values to determine which flash flooding events were the most expensive
highest_damage = df.sort_values("DAMAGE_PROPERTY_NUM", ascending=False)

print(highest_damage[
    ["BEGIN_DATE", "COUNTY_NAME", "DAMAGE_PROPERTY_NUM"]
].head(10))

In [ ]:
# Sort Values to determine which flash flooding events had the most injuries
most_injuries = df.sort_values("INJURIES_DIRECT", ascending=False)

print(most_injuries[
    ["BEGIN_DATE", "COUNTY_NAME", "INJURIES_DIRECT"]
].head(10))

In [ ]:
# Create a summary table for the following question: What were the causes of flash floods?
cause_table = (
    df.groupby("FLOOD_CAUSE")
      .size()
      .reset_index(name="Count")
      .sort_values("Count", ascending=False)
)

print(cause_table.head(10))

In [ ]:
# Create a summary table for the following question: What has been the total statewide impact of flash flooding in Kentucky from 2015-2025?
summary = pd.DataFrame({
    "Total Events": [len(df)],
    "Total Property Damage": [df["DAMAGE_PROPERTY_NUM"].sum()],
    "Total Deaths": [df["DEATHS_DIRECT"].sum()],
    "Total Injuries": [df["INJURIES_DIRECT"].sum()]
})

print(summary)

In [ ]:
# Create a summary table for the following question: How has reporting sources changed over time?
source_year = pd.crosstab(df["YEAR"], df["SOURCE"])

print(source_year)

In [ ]:
# Create a heatmap that shows correlations between numerical variables
cols = [
    "YEAR",
    "DEATHS_DIRECT",
    "INJURIES_DIRECT",
    "DAMAGE_PROPERTY_NUM",
    "DAMAGE_CROPS_NUM"
]

corr_matrix = df[cols].corr()

plt.figure(figsize=(10, 6))

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Matrix of Selected Flash Flood Variables")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=30, va="center")

plt.show()

In [ ]:
# Create a heatmap that shows correlations between years and sources
source_year = pd.crosstab(df["YEAR"], df["SOURCE"]) 

plt.figure(figsize=(14, 10))

sns.heatmap(
    source_year,
    annot=True,
    cmap="YlGnBu",
    fmt="d"
)

plt.title("Flash Flood Events by Year and Source")
plt.xlabel("Source")
plt.ylabel("Year")
plt.xticks(rotation=60, ha="right")
plt.yticks()

plt.show()